# Phase 6: Error Analysis and Experiment Transfer

This phase investigates where Phase 5 predictions succeed or fail before considering higher-capacity models. It preserves the Phase 5 grouped validation methods and Random Forest configuration. Outputs describe Hiltunen experimental Chlorella cultures, not a deployed Liquid Tree.

## 1. Reproduce Phase 5 first
Re-run the existing evaluation in memory and compare it with the saved summary and fold CSVs. This check is read-only: it does not call the Phase 5 export function or rewrite its artifacts. Additional analyses run only after this comparison passes.

In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd

CANDIDATE_PATH = Path("data/processed/hiltunen_next_day_density_basic.csv")
search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next((path for path in search_locations if (path / CANDIDATE_PATH).is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Could not find {CANDIDATE_PATH} from this directory or its parents.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.models.error_analysis as error_analysis

error_analysis = importlib.reload(error_analysis)
phase5_reproduction = error_analysis.reproduce_phase5(PROJECT_ROOT)
print("Phase 5 reproduction:", phase5_reproduction["reproduced"])
print("Maximum summary metric difference:", phase5_reproduction["max_summary_metric_delta"])
print("Maximum fold metric difference:", phase5_reproduction["max_fold_metric_delta"])
print("Phase 5 outputs written:", phase5_reproduction["phase5_outputs_written"])
display(phase5_reproduction["results"][["Feature_Set", "Model", "Evaluation_Protocol", "MAE", "RMSE", "R2"]])

## 2. Held-out experiment and condition analysis
Only after the Phase 5 reproduction passes, run the reusable error-analysis pipeline. For each experiment the condition/context table reports culture/sample counts, treatment labels, density distributions, mean observed day-to-day change, and differences from the pooled remaining experiments. These differences are descriptive, not causal.

In [ ]:
analysis = error_analysis.run_error_analysis(PROJECT_ROOT, reproduction=phase5_reproduction)
experiment_conditions = analysis["experiment_conditions"]
experiment_results = analysis["experiment_level"]
predictions = analysis["predictions"]
error_by_day = analysis["error_by_day"]
error_by_density_bin = analysis["error_by_density_bin"]
feature_importance = analysis["feature_importance"]
reduced_results = analysis["reduced_features"]
ph_comparison = analysis["ph_comparison"]
print("Experiment condition and target/current distribution summary:")
display(experiment_conditions)
print("All Phase 5 leave-one-experiment-out fold metrics with experiment context:")
display(experiment_results[["Exp", "Feature_Set", "Model", "N_Validation", "MAE", "RMSE", "R2", "Current_Density_Mean_Difference_vs_Training", "Target_Density_Mean_Difference_vs_Training"]])

## 3. Random Forest out-of-fold predictions
Each row below is a held-out prediction from the original Phase 5 grouped folds. Residual is actual minus predicted; these predictions are for diagnostics only and are not reused as training data.

In [ ]:
print("OOF rows, feature sets, and protocols:", predictions.shape, predictions["Feature_Set"].value_counts().to_dict(), predictions["Evaluation_Protocol"].value_counts().to_dict())
display(predictions.head(12))
print("Fold-level RF scores were checked against Phase 5:")
display(analysis["rf_fold_metrics"].head(10))

## 4. Error by current day
Sample counts are included in every day summary. Later-day errors with only four samples (days 13-15) are especially uncertain and should not be treated as stable phase-specific performance.

In [ ]:
display(error_by_day)
print("Phase 4 eligible next-day counts:")
display(pd.read_csv(PROJECT_ROOT / "data/processed/hiltunen_next_day_pairs_by_day.csv")[["Day", "Valid_Next_Day_Pairs"]])

## 5. Error by current-density quantile
Four quantile bins are formed from the evaluated OOF current-density values separately within each protocol and feature set. Bin edges are data-derived for descriptive analysis, not tuned decision thresholds.

In [ ]:
display(error_by_density_bin)

## 6. Predicted/actual and residual plots
The dashed diagonal is the equality line. Residuals are actual minus predicted; positive residuals indicate underprediction.

In [ ]:
from IPython.display import Image, display

for filename in [
    "rf_predicted_vs_actual_GroupKFold_Culture_5.png",
    "rf_predicted_vs_actual_LeaveOneExperimentOut.png",
    "rf_residual_vs_predicted.png",
    "rf_mae_by_day.png",
    "rf_mae_by_heldout_experiment.png",
]:
    print(filename)
    display(Image(filename=str(analysis["figure_directory"] / filename)))

## 7. Feature importance
Impurity importance sums one-hot dummy importances back to the original condition variable. Permutation importance is measured on held-out data as the increase in MAE after permutation. Neither establishes a causal or biological effect.

In [ ]:
display(feature_importance)

## 8. Contribution beyond current density
Random Forest is compared with current density only, current density plus day, and the complete basic condition feature set. The persistence baseline is scored on the same folds. This is a targeted feature contribution check, not a tuned competition.

In [ ]:
display(reduced_results)

## 9. Paired pH comparison
Basic and pH metrics are compared on matched folds. Differences are pH minus basic: negative MAE/RMSE differences favor pH; positive R2 differences favor pH. Fold variation is included.

In [ ]:
display(ph_comparison)

## 10. Temporal coverage and generalization
Unseen-culture evaluation asks whether prediction transfers to a new culture within experimental contexts represented in training. Unseen-experiment evaluation withholds all cultures in one experiment and is a harder, more independent transfer test. The four held-out experiments are too few for a stable deployment claim.

Late-day availability falls to 29 samples for days 10-12 and four for days 13-15. This supports descriptive checks but not a meaningful deep temporal model assessment. Hiltunen is experimental Chlorella data, not data from the deployed Liquid Tree sensors.

## 11. Limitations
Only four independent experiments contribute repeated culture-level observations. Late-day sample loss makes late-day errors unstable. The experiments differ in treatment composition and observed density distributions; these associations do not establish causal explanations. There is no external validation dataset or actual deployed Liquid Tree sensor dataset in this phase.

## 12. Recommendation
The results do not justify ANN/LSTM/GRU yet. Current density dominates the Random Forest importance summaries, reduced features show that day and treatment features add uneven value across protocols, and unseen-experiment errors vary substantially. Acquire more independent experiments and validate on actual Liquid Tree sensor data before increasing model complexity. No Phase 7 work is included.